In [4]:
import numpy as np
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

query = "What is the capital of France?"
passages = [
    "London is the capital of the United Kingdom.",
    "Paris is the capital and most populous city of France.",
    "Python is a popular programming language.",
    "Every country has its own capital city, which serves as the administrative center. France also has its own capital, which is known for its rich history and culture.",
]

pairs = [[query, p] for p in passages]
scores = reranker.predict(pairs)

print(scores)


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


normalized_scores = sigmoid(scores)
results = sorted(zip(normalized_scores, passages), reverse=True)

for score, passage in results:
    print(f"{score:.4f} -> {passage}")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 8453.22it/s]


[5.4991362e-03 9.9935907e-01 1.6127056e-05 3.4617683e-01]
0.7309 -> Paris is the capital and most populous city of France.
0.5857 -> Every country has its own capital city, which serves as the administrative center. France also has its own capital, which is known for its rich history and culture.
0.5014 -> London is the capital of the United Kingdom.
0.5000 -> Python is a popular programming language.


In [5]:
from sentence_transformers import SentenceTransformer

# Bi-encoder for the diversity term: MMR needs passage-vs-passage similarity,
# which the cross-encoder can't give (it only scores query-passage pairs).
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

query = "How does self-attention work in the Transformer?"
passages = [
    "Self-attention computes a weighted sum of the value vectors, where each weight is the softmax of the dot product between the query and key, scaled by 1/sqrt(d_k).",
    "Scaled dot-product attention takes the dot products of the query with all keys, divides each by sqrt(d_k), and applies a softmax to get the weights on the values.",
    "Attention is computed as softmax(QK^T / sqrt(d_k))V, the scaled dot-product attention used throughout the Transformer.",
    "Multi-head attention runs several attention functions in parallel on linearly projected queries, keys, and values, then concatenates and projects the results.",
    "Positional encodings are added to the input embeddings so the model can use sequence order, since attention on its own is permutation-invariant.",
    "Training used the Adam optimizer with a warmup schedule that increases the learning rate linearly for the first few thousand steps.",
]

# relevance term -> reranker score (reusing `reranker` and `sigmoid` from above)
relevance = sigmoid(reranker.predict([[query, p] for p in passages]))

# diversity term -> cosine between passage vectors (normalized, so dot == cosine)
vectors = embedder.encode(passages, normalize_embeddings=True)
similarity = vectors @ vectors.T


def mmr(relevance, similarity, k, lam):
    selected, remaining = [], list(range(len(relevance)))
    while remaining and len(selected) < k:
        if not selected:
            best = max(remaining, key=lambda i: relevance[i])
        else:
            best = max(
                remaining,
                key=lambda i: (
                    lam * relevance[i]
                    - (1 - lam) * max(similarity[i][j] for j in selected)
                ),
            )
        selected.append(best)
        remaining.remove(best)
    return selected


k = 3
rerank_only = sorted(range(len(passages)), key=lambda i: relevance[i], reverse=True)[:k]
rerank_mmr = mmr(relevance, similarity, k=k, lam=0.6)

print("Reranker only — top 3 by relevance:")
for i in rerank_only:
    print(f"  {relevance[i]:.3f}  {passages[i][:75]}")

print("\nReranker -> MMR (lambda=0.6) — top 3:")
for i in rerank_mmr:
    print(f"  {relevance[i]:.3f}  {passages[i][:75]}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16409.45it/s]


Reranker only — top 3 by relevance:
  0.658  Self-attention computes a weighted sum of the value vectors, where each wei
  0.556  Attention is computed as softmax(QK^T / sqrt(d_k))V, the scaled dot-product
  0.504  Positional encodings are added to the input embeddings so the model can use

Reranker -> MMR (lambda=0.6) — top 3:
  0.658  Self-attention computes a weighted sum of the value vectors, where each wei
  0.500  Training used the Adam optimizer with a warmup schedule that increases the 
  0.504  Positional encodings are added to the input embeddings so the model can use
